# Teil 3 - Modelltraining

In diesem Notebook bereite ich Features und Zielvariable vor, teile die Daten in Trainings- und Testdaten auf, wähle einen passenden sklearn-Algorithmus und trainiere ein Modell. Das Ziel ist die Vorhersage, ob eine Person hohen Stress hat (`High_Stress = 1`).

## 1. Daten laden und Zielvariable bilden

`Stress_Level` liegt im Datensatz auf einer Skala von 1 bis 10 vor. Für ein klares Klassifikationsproblem definiere ich `High_Stress` als 1, wenn `Stress_Level >= 7` ist, sonst 0.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_FILE = Path('Smartphone_Usage_Productivity_Dataset_50000.csv')
df = pd.read_csv(DATA_FILE, sep=';')
df['High_Stress'] = (df['Stress_Level'] >= 7).astype(int)

df[['Stress_Level', 'High_Stress']].head(10)

,Stress_Level,High_Stress
0,4,0
1,1,0
2,4,0
3,3,0
4,3,0
5,7,1
6,3,0
7,3,0
8,4,0
9,2,0


In [2]:
df['High_Stress'].value_counts(normalize=True).rename(index={0: 'kein hoher Stress', 1: 'hoher Stress'}).round(3)

High_Stress
kein hoher Stress    0.601
hoher Stress         0.399
Name: proportion, dtype: float64

## 2. Features vorbereiten

Die `User_ID` wird entfernt, weil sie nur eine eindeutige Kennung ist und keine fachliche Information für neue Personen liefert. `Stress_Level` wird ebenfalls entfernt, weil daraus die Zielvariable gebildet wurde. Numerische und kategoriale Spalten werden getrennt verarbeitet.

In [3]:
X = df.drop(columns=['User_ID', 'Stress_Level', 'High_Stress'])
y = df['High_Stress']

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [column for column in X.columns if column not in numeric_features]

print('Numerische Features:', numeric_features)
print('Kategoriale Features:', categorical_features)

Numerische Features: ['Age', 'Daily_Phone_Hours', 'Social_Media_Hours', 'Work_Productivity_Score', 'Sleep_Hours', 'App_Usage_Count', 'Caffeine_Intake_Cups', 'Weekend_Screen_Time_Hours']
Kategoriale Features: ['Gender', 'Occupation', 'Device_Type']


## 3. Fehlende Werte und nicht-numerische Spalten behandeln

Obwohl der Datensatz aktuell keine fehlenden Werte hat, baue ich einen sauberen `Pipeline`-Ablauf. Numerische Werte werden bei Bedarf mit dem Median ersetzt. Kategoriale Werte werden mit dem häufigsten Wert ersetzt und anschliessend per One-Hot-Encoding in numerische Modellspalten umgewandelt.

In [4]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 4. Train-/Test-Aufteilung

Ich verwende 80 Prozent der Daten für Training und 20 Prozent für Test. Mit `stratify=y` bleibt der Anteil von hohem und nicht hohem Stress in beiden Teilen ungefähr gleich.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Trainingsdaten:', X_train.shape)
print('Testdaten:', X_test.shape)
print('Anteil hoher Stress im Training:', round(y_train.mean(), 3))
print('Anteil hoher Stress im Test:', round(y_test.mean(), 3))

Trainingsdaten: (40000, 11)
Testdaten: (10000, 11)
Anteil hoher Stress im Training: 0.399
Anteil hoher Stress im Test: 0.399


## 5. Algorithmuswahl

Ich verwende einen `RandomForestClassifier`, weil das Ziel eine Klassifikation ist und die Daten aus numerischen sowie kategorialen Feldern bestehen. Random Forests können nichtlineare Zusammenhänge abbilden und liefern später Feature Importances, was für Teil 4 nützlich ist. `class_weight='balanced'` verhindert, dass das Modell wegen der Klassenverteilung nur die grössere Klasse bevorzugt. Die Tiefe wird begrenzt, damit das Modell weniger stark überfitten kann.

In [6]:
model = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', model)
])

## 6. Modell trainieren

Das Modell lernt nur auf den Trainingsdaten. Die Testdaten bleiben bis zur Vorhersage unberührt, damit die Bewertung fair bleibt.

In [7]:
clf.fit(X_train, y_train)
print('Modell wurde trainiert.')

Modell wurde trainiert.


## 7. Einige Vorhersagen prüfen

Ich lasse das Modell für einige Testzeilen vorhersagen, ob hoher Stress vorliegt. Die Wahrscheinlichkeit für Klasse 1 hilft bei der manuellen Einschätzung, ob das Modell sehr sicher oder eher unsicher ist.

In [8]:
predictions = clf.predict(X_test)
probabilities = clf.predict_proba(X_test)[:, 1]

check = X_test.copy()
check['Stress_Level_echt'] = df.loc[X_test.index, 'Stress_Level']
check['High_Stress_echt'] = y_test
check['High_Stress_vorhersage'] = predictions
check['Wahrscheinlichkeit_High_Stress'] = probabilities

check[[
    'Daily_Phone_Hours',
    'Social_Media_Hours',
    'Sleep_Hours',
    'Weekend_Screen_Time_Hours',
    'Stress_Level_echt',
    'High_Stress_echt',
    'High_Stress_vorhersage',
    'Wahrscheinlichkeit_High_Stress'
]].sample(10, random_state=7).round(3)

,Daily_Phone_Hours,Social_Media_Hours,Sleep_Hours,Weekend_Screen_Time_Hours,Stress_Level_echt,High_Stress_echt,High_Stress_vorhersage,Wahrscheinlichkeit_High_Stress
37631,8.4,8.0,5.8,4.7,5,0,1,0.501
20959,8.5,6.6,7.6,8.4,2,0,0,0.500
18262,7.2,4.2,9.0,8.7,10,1,0,0.497
22510,9.5,6.4,4.6,8.7,2,0,0,0.485
28147,5.4,3.2,5.5,13.0,3,0,0,0.493
49175,9.7,1.6,6.5,10.8,1,0,1,0.509
11557,9.0,3.7,8.8,10.5,8,1,1,0.504
21570,2.0,1.7,8.5,11.4,9,1,1,0.506
1411,5.8,0.6,4.7,11.4,5,0,0,0.499
18147,10.2,7.1,6.1,13.0,8,1,0,0.499


In [9]:
print('Accuracy:', round(accuracy_score(y_test, predictions), 3))
print('Precision:', round(precision_score(y_test, predictions), 3))
print('Recall/Sensitivität:', round(recall_score(y_test, predictions), 3))
print('F1-Score:', round(f1_score(y_test, predictions), 3))
print('Confusion Matrix:')
print(confusion_matrix(y_test, predictions))

Accuracy: 0.514
Precision: 0.409
Recall/Sensitivität: 0.489
F1-Score: 0.446
Confusion Matrix:
[[3193 2817]
 [2039 1951]]


## Zusammenfassung Teil 3

Das Modell ist technisch korrekt trainiert und liefert nachvollziehbare Vorhersagen, aber die manuelle Prüfung zeigt viele unsichere Fälle. Die Wahrscheinlichkeiten liegen oft nahe bei 0.5, was zu den sehr schwachen Korrelationen aus Teil 2 passt. Deshalb ist das Modell nicht stark genug für verlässliche echte Entscheidungen. Für das Schulprojekt ist dieser Befund trotzdem wertvoll, weil er zeigt, dass Datenmenge allein keine gute Vorhersage garantiert.